In [ ]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
import os, json, math
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# from dotenv import load_dotenv
# load_dotenv()
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
MODEL = "gpt-4o-mini"

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool, StructuredTool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

llm = ChatOpenAI(model="gpt-4o-mini")

# 오늘은 Tool function calling 등을 다룰 예정

In [ ]:
# 서울 날씨 알려줘? -> LLM -> tool_calls -> 개발자가 만든 get_weather("서울") -> 툴에 대한 결과(result) -> 답변: 서울은 맑아요
  # LLM은 실시간 데이터에 대한 접근이, 모델만으로는 불가

In [ ]:
def get_weather(city):
  weather_db = {
      "seoul" : "맑음, 22도, 습도 45%",
      "tokyo" : "흐림, 19도, 습도 70%",
      "new york" : "비, 15도, 습도 80%"
  }
  return weather_db.get(city, f"{city}: 정보 없음")

In [ ]:
# 이런식으로 tool 등을 호출한다는 예시 코드임 (그냥 목업 함수 이용)
tool_calls = {"name": "get_weather", "args": {"city" : "seoul"}}
result = get_weather(**tool_calls['args'])

In [ ]:
result

'맑음, 22도, 습도 45%'

In [ ]:
# langchain에서는 @tool 데코레이터를 사용하면
  # 해당 데코레이터로 정의되어있는 것들을 찾도록 함
@tool
def get_weather(city):
  """날씨를 조회하는 함수입니다."""
  weather_db = {
      "seoul" : "맑음, 22도, 습도 45%",
      "tokyo" : "흐림, 19도, 습도 70%",
      "new york" : "비, 15도, 습도 80%"
  }
  return weather_db.get(city, f"{city}: 정보 없음")

In [ ]:
get_weather.name

'get_weather'

In [ ]:
get_weather.description

'날씨를 조회하는 함수입니다.'

In [ ]:
get_weather.args

{'city': {'title': 'City'}}

In [ ]:
llm_with_tools = llm.bind_tools([get_weather])

In [ ]:
result = llm_with_tools.invoke("서울 날씨 어때") # 출력에서 "tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}," 확인가능
result # content가 비어있는 것을 볼수있음

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 50, 'total_tokens': 64, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_998d5473a0', 'id': 'chatcmpl-DX3LakxShS99XPuJ96hozMBVbEUKp', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dafbf-982a-76e0-b5be-179593fb6c68-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_ntshz72MLqaSr9DE7HW7QDiF', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 14, 'total_tokens': 64, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
result.tool_calls

[{'name': 'get_weather',
  'args': {'city': '서울'},
  'id': 'call_sZ6WwEPBxQ5zKdgkvjcYK23R',
  'type': 'tool_call'}]

In [ ]:
result2 = llm_with_tools.invoke("파이썬이 뭐야?")
result2 # tool_calls가 비어있는 것을 볼수있음

AIMessage(content='파이썬(Python)은 높은 수준의 프로그래밍 언어로, 코드가 간결하고 읽기 쉬운 특징이 있습니다. 1991년 귀도 반 로썸(Guido van Rossum)이 처음 개발하였으며, 다양한 프로그래밍 패러다임을 지원합니다. 주요 특징은 다음과 같습니다:\n\n1. **가독성**: 파이썬은 문법이 간단하여 코드가 읽기 쉽고 이해하기 용이합니다.\n2. **다양한 라이브러리**: 과학 계산, 데이터 분석, 웹 개발 등 다양한 분야에 사용할 수 있는 풍부한 라이브러리를 제공합니다.\n3. **인터프리터 언어**: 파이썬은 인터프리터 언어로, 작성한 코드를 즉시 실행할 수 있으며, 빠른 프로토타이핑에 유리합니다.\n4. **커뮤니티**: 활발한 사용자 커뮤니티가 있어, 도움을 받을 수 있는 자료와 지원이 많습니다.\n\n파이썬은 웹 개발(예: Django, Flask), 데이터 분석(예: Pandas, NumPy), 인공지능 및 머신러닝(예: TensorFlow, Keras) 등 다양한 분야에서 널리 사용되고 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 270, 'prompt_tokens': 53, 'total_tokens': 323, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_998d5473a0', 'id': 'cha

In [ ]:
if result.tool_calls:
  tc = result.tool_calls[0]
  print(f"tool: {tc['name']}")

  result_final = get_weather.invoke(tc['args'])
  print(f"result: {result_final}") # 'seoul' 이라고 정의되어 있어서 정보없음 이라고 뜸

tool: get_weather
result: 서울: 정보 없음


In [ ]:
# 질문을 받아서 llm이 bind된 툴을 찾고, 툴이 필요있는지 없는지 검토 -> 포맷을 채워 tool을 실행하는데, 이때 llm이 자체적으로 포맷을 채움
# 따라서 tool 실행에 따른 토큰 소모가 추가로 됨

In [ ]:
@tool
def get_exchange_rate(from_currency: str, to_currency: str) -> str:
  """환율을 알려주는 함수입니다. from_currency= 원, 달러, 엔 등으로 입력하세요"""
  currency_db = {
      "달러-원": "1500원",
      "원-달러": "1000원당 0.8달러",
      "달러-엔": "1달러당 150엔",
      "엔-달러": "100엔당 0.7달러",
      "엔-원": "100엔당 900원",
      "원-엔": "1000원당 111엔"
  }

  query_key = f"{from_currency}-{to_currency}"
  return currency_db.get(query_key, f"{query_key}: 정보 없음")

llm_with_tools = llm.bind_tools([get_exchange_rate])
llm_with_tools.invoke("원-달러 환율 알려줘.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 75, 'total_tokens': 97, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4debc47fe0', 'id': 'chatcmpl-DX3aQalCe5gojVNUTePxfaAB5o4lE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dafcd-a0c8-78c1-9fb0-447574e1b2c9-0', tool_calls=[{'name': 'get_exchange_rate', 'args': {'from_currency': '원', 'to_currency': '달러'}, 'id': 'call_ffn4buoPR8F0r4q6tTHXEJUB', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 22, 'total_tokens': 97, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, '

In [ ]:
# 강사님 코드
@tool
def get_exchange_rate(from_currency: str, to_currency: str) -> str:
  """두 통화 간의 환율을 조회합니다."""
  currency_db = {
      ("KRW", "USD") : 0.00080, ("KRW", "JPY") : 0.11,
      ("USD", "KRW") : 1400, ("JPY", "KRW") : 9.0
  }

  return f"1 {from_currency} = {currency_db.get((from_currency, to_currency))} {to_currency}"

llm_with_tool = llm.bind_tools([get_exchange_rate])
response = llm_with_tool.invoke("원-달러 환율 알려줘.")

In [ ]:
response.tool_calls

[{'name': 'get_exchange_rate',
  'args': {'from_currency': 'KRW', 'to_currency': 'USD'},
  'id': 'call_Ai7HdqJov3S9C7nCwAenTqEC',
  'type': 'tool_call'}]

In [ ]:
if response.tool_calls:
  tc = response.tool_calls[0]
  result = tc['name'].invoke(tc['args'])

result

AttributeError: 'str' object has no attribute 'invoke'

In [ ]:
type(get_exchange_rate)

langchain_core.tools.structured.StructuredTool

In [ ]:
tools = [get_exchange_rate]
tool_map = {t.name: t for t in tools}

if response.tool_calls:
  tc = response.tool_calls[0]
  tool = tool_map.get(tc['name'])

In [ ]:
get_exchange_rate.invoke({'from_currency':'KRW', 'to_currency': 'USD' })

'1 KRW = 0.0008 USD'

In [ ]:
# StructuredTool.from_function
def search_etf_func(category, min_return, max_expense=1.0):
  etfs = [
        {"name": "KODEX 200", "cat": "국내주식", "ret": 8.5, "exp": 0.15},
        {"name": "KODEX S&P500TR", "cat": "해외주식", "ret": 25.3, "exp": 0.05},
        {"name": "TIGER 나스닥100", "cat": "해외주식", "ret": 30.2, "exp": 0.07},
        {"name": "ACE 미국배당다우존스", "cat": "배당", "ret": 12.1, "exp": 0.01},
    ]

  result = [e for e in etfs if e['cat'] == category and e['ret'] >= min_return and e['exp'] <= max_expense]
  return json.dumps(result, ensure_ascii = False) if result else '조건에 맞는 ETF 없음'

In [ ]:
from pydantic import BaseModel, Field

class ETFSearchinput(BaseModel):
  category : str = Field(description= "ETF 카테고리")
  min_return : float = Field(default =0, le=100, description = '최소 수익률')
  max_expense : float = Field(default=1.0, ge=0, le=5, description = '최대 운용보수')

In [ ]:
search_etf = StructuredTool.from_function(
    func = search_etf_func, name = 'search_etf',
    description = '조건에 맞는 상품을 검색합니다.',
    args_schema = ETFSearchinput,
)

search_etf

StructuredTool(name='search_etf', description='조건에 맞는 상품을 검색합니다.', args_schema=<class '__main__.ETFSearchinput'>, func=<function search_etf_func at 0x7f84f3dbb2e0>)

In [ ]:
search_etf.name, search_etf.args

('search_etf',
 {'category': {'description': 'ETF 카테고리',
   'title': 'Category',
   'type': 'string'},
  'min_return': {'default': 0,
   'description': '최소 수익률',
   'maximum': 100,
   'title': 'Min Return',
   'type': 'number'},
  'max_expense': {'default': 1.0,
   'description': '최대 운용보수',
   'maximum': 5,
   'minimum': 0,
   'title': 'Max Expense',
   'type': 'number'}})

In [ ]:
llm_etf = llm.bind_tools([search_etf])

In [ ]:
resp # tool_calls에 출력이 나와있음

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 120, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e61ea1dda4', 'id': 'chatcmpl-DX3w85XwLUVdEUhIiga01HawS4f89', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dafe2-2b05-7422-86aa-f2113f8b0cea-0', tool_calls=[{'name': 'search_etf', 'args': {'category': '배당', 'max_expense': 0.5}, 'id': 'call_0kWmhWm5Eb60pAe8Wi6ZA7MH', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 120, 'output_tokens': 24, 'total_tokens': 144, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasonin

In [ ]:
queries = ['해외주식 ETF 추천해줘', '수수료 낮은 배당 ETF 찾아줘']
for q in queries:
  resp = llm_etf.invoke(q)
  if resp.tool_calls:
    tc = resp.tool_calls[0]
    result = search_etf.invoke(tc['args'])
    print(f"Q: {q}")
    print(tc['args'])
    print(f"A: {result}")

Q: 해외주식 ETF 추천해줘
{'category': '해외주식'}
A: [{"name": "KODEX S&P500TR", "cat": "해외주식", "ret": 25.3, "exp": 0.05}, {"name": "TIGER 나스닥100", "cat": "해외주식", "ret": 30.2, "exp": 0.07}]
Q: 수수료 낮은 배당 ETF 찾아줘
{'category': '배당', 'max_expense': 0.5}
A: [{"name": "ACE 미국배당다우존스", "cat": "배당", "ret": 12.1, "exp": 0.01}]


In [ ]:
from pydantic import BaseModel, Field

class EmployeeSearchInput(BaseModel):
  department : str = Field(description= "부서명", enum=['개발팀', '마케팅', '영업'])
  min_salary: int = Field(default=3000, ge=3000, description="최소 연봉(만원)")

def search_employee(department, min_salary=3000):
  """최소 월급 이상을 받는 직원의 목록을 검색하는 함수."""
  employees = [
      {"name": "가나다", "department": "개발팀", "salary": 6000},
      {"name": "나다가", "department": "마케팅", "salary": 3500},
      {"name": "홍길동", "department": "마케팅", "salary": 4000},
      {"name": "김철수", "department": "영업", "salary": 5000},
  ]

  # found_employees = []
  # for emp in employees:
  #     if emp["department"] == department and emp["salary"] >= min_salary:
  #         found_employees.append(emp["name"])

  result = [e for e in employees if e['department'] == department and e['salary'] >= min_salary]
  return result

emp_tool = StructuredTool.from_function(
    func = search_employee, name = 'search_employee',
    description = '부서와 연봉 조건으로 직원 검색.',
    args_schema = EmployeeSearchInput,
)

llm_emp = llm.bind_tools([emp_tool])
resp = llm_emp.invoke("연봉 4000 이상 개발자 보여주세요")

/tmp/ipykernel_10671/2140880400.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'enum'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  department : str = Field(description= "부서명", enum=['개발팀', '마케팅', '영업'])


In [ ]:
if resp.tool_calls:
  result = emp_tool.invoke(resp.tool_calls[0]['args'])
  print(resp.tool_calls[0])
  print(result)

{'name': 'search_employee', 'args': {'department': '개발', 'min_salary': 4000}, 'id': 'call_g5fvNKagfbs8zvDFtFVeAM96', 'type': 'tool_call'}
[]


In [ ]:
# 여러개의 툴을 이용하는 것
  # get_weather_tool : 날씨
  # get_exchange_rate : 환율
  # 서울과 도쿄 날씨 비교하고 환율도 알려줘

# tool_calls: {
#     {}
# }

In [ ]:
all_tools = [get_weather, get_exchange_rate, search_etf]
llm_multi = llm.bind_tools(all_tools) # 위의 툴들을 모두 llm에 바인딩

In [ ]:
resp = llm_multi.invoke("서울(seoul)과 도쿄(tokyo) 날씨 비교하고 환율도 알려줘")

resp
""" 아래 처럼 tool 호출된 것 확인 가능
tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_OIyGr6IG7kuwpBo4TK1iiakS', 'type': 'tool_call'},
 {'name': 'get_weather', 'args': {'city': '도쿄'}, 'id': 'call_zWClqLwhFZgZBANIph7SU5t7', 'type': 'tool_call'},
 {'name': 'get_exchange_rate', 'args': {'from_currency': 'KRW', 'to_currency': 'JPY'}, 'id': 'call_5LArBWmGN2qNhkgx3dkdRJYX', 'type': 'tool_call'}]
"""

" 아래 처럼 tool 호출된 것 확인 가능\ntool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_OIyGr6IG7kuwpBo4TK1iiakS', 'type': 'tool_call'},\n {'name': 'get_weather', 'args': {'city': '도쿄'}, 'id': 'call_zWClqLwhFZgZBANIph7SU5t7', 'type': 'tool_call'}, \n {'name': 'get_exchange_rate', 'args': {'from_currency': 'KRW', 'to_currency': 'JPY'}, 'id': 'call_5LArBWmGN2qNhkgx3dkdRJYX', 'type': 'tool_call'}]\n"

In [ ]:
tool_map = {t.name: t for t in all_tools}
for tc in resp.tool_calls:
  result = tool_map[tc['name']].invoke(tc['args'])
  print(f" {tc['name']} {tc['args']} -> {result}")

# tool에 인자만 넣어서 결과를 뽑은것
# 즉 RAG로 치면 문서 검색 후 반환만 한 것

 get_weather {'city': 'seoul'} -> 맑음, 22도, 습도 45%
 get_weather {'city': 'tokyo'} -> 흐림, 19도, 습도 70%
 get_exchange_rate {'from_currency': 'KRW', 'to_currency': 'JPY'} -> 1 KRW = 0.11 JPY


In [ ]:
# 실습
  # 1. 계산기 툴을 추가
  # 2. [get_weather, get_exchange_rate, 계산기]를 바인드하고
  # 3. 질문에 대한 답을 받기 ("서울 날씨 알려주고 15*24 계산해줘.")

def calculator(a: float, b: float, operator: str) -> float:
    """
    사칙연산 계산기
    """
    if operator == '+': return a + b
    elif operator == '-': return a - b
    elif operator == '*': return a * b
    elif operator == '/': return a / b
    else: return "지원하지 않는 연산자"


all_tools_calc = [get_weather, get_exchange_rate, calculator]
llm_multi = llm.bind_tools(all_tools_calc) # 위의 툴들을 모두 llm에 바인딩

In [ ]:
from langchain_core.tools import tool

@tool
def calculate(expression):
  """수학 계산식을 평가합니다."""
  try:
    result = eval(expression)
    return f"{expression} = {result}"
  except:
    return 'error'

tools = [get_weather, get_exchange_rate, calculate]
llm_3 = llm.bind_tools(tools)
tool_map_3 = {t.name:t for t in tools}
resp = llm_3.invoke("서울 날씨 알려주고 15*24 계산해줘")
display(resp)

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 110, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DX4hIpOQTAY13xwP77EubmjJad6sJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db00e-c986-79a3-a810-7fb6cc450ac9-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_bkNjzyF110HSDLWdmKcagATc', 'type': 'tool_call'}, {'name': 'calculate', 'args': {'expression': '15*24'}, 'id': 'call_sYr1zXB4NUhI1EqaXp65aJsj', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 110, 'output_tokens': 45, 'total_tokens': 155, 'input_

In [ ]:
all_tools = [get_weather, get_exchange_rate]
llm_with_tools = llm.bind_tools(all_tools)

In [ ]:
query = "서울 날씨 알려주고, 원-엔 환율도 알려줘"

In [ ]:
messages = [HumanMessage(content=query)]
ai_msg = llm_with_tools.invoke(messages)

messages.append(ai_msg)

tool_map = {t.name:t for t in all_tools}

for tc in ai_msg.tool_calls:
  selected_tool = tool_map[tc['name']]
  tool_result = selected_tool.invoke(tc['args'])

  messages.append(ToolMessage(content = str(tool_result), tool_call_id = tc['id']))

In [ ]:
ai_msg

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 92, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_df1291c4ff', 'id': 'chatcmpl-DX4j0ws1yxCzeBde66IjQRAZSxzcq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db010-643c-77a2-854f-a97e24d7554c-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_oqyrFBhqQzt4ZacGljXqoOgV', 'type': 'tool_call'}, {'name': 'get_exchange_rate', 'args': {'from_currency': 'KRW', 'to_currency': 'JPY'}, 'id': 'call_LHsmhRAtN5XR4IcNDc4GAPE4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 92, 'output_tokens': 52,

In [ ]:
messages

[HumanMessage(content='서울 날씨 알려주고, 원-엔 환율도 알려줘', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 92, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_df1291c4ff', 'id': 'chatcmpl-DX4m2oort6F8mpzgRAnRsLxd6bZhN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db013-46c9-7810-a574-7d0717de0098-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '서울'}, 'id': 'call_MzRkKvjCEYVXRUOlkG6yFxZh', 'type': 'tool_call'}, {'name': 'get_exchange_rate', 'args': {'from_currency': 'KRW', 'to_currency': 'JPY'}, 'id': 'call_VhQ7s774A1ecWMXra0bGadDF', 'type':

In [ ]:
final_answer = llm_with_tools.invoke(messages)
final_answer

AIMessage(content='서울의 날씨 정보는 현재 확인할 수 없습니다. 하지만 원-엔 환율은 1 KRW가 0.11 JPY임을 알려드립니다. 추가로 궁금한 점이 있으신가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 172, 'total_tokens': 220, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_df1291c4ff', 'id': 'chatcmpl-DX4nHgWOrIy7eCexZ6isOwEfu3ghc', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019db014-7092-7162-b8d2-cc25c68b0be4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 172, 'output_tokens': 48, 'total_tokens': 220, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
final_answer.content

'서울의 날씨 정보는 현재 확인할 수 없습니다. 하지만 원-엔 환율은 1 KRW가 0.11 JPY임을 알려드립니다. 추가로 궁금한 점이 있으신가요?'

In [ ]:
def tool_loop(query, tools, max_turns = 5): # trade-off, 예외처리, 방어로직(무한루프 돌면서 토큰 갉아먹는 문제..)

  tool_map = {t.name:t for t in tools}
  llm_t = llm.bind_tools(tools)
  messages = [HumanMessage(content = query)]

  for turn in range(max_turns):
    response = llm_t.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
      return response # 툴을 호출 안했으니, 그냥 리턴

    for tc in response.tool_calls:
      tool_obj = tool_map[tc['name']]
      result = tool_obj.invoke(tc['args']) if tool_obj else "도구 없음."
      messages.append(ToolMessage(content = str(result), tool_call_id = tc['id']))
      print(f" [Turn {turn+1}] {tc['name']}, {tc['args']} -> {result}")


  return "최대 턴 초과"


In [ ]:
ans = tool_loop("서울 날씨 알려주고, 원-엔 환율도 알려줘", [get_weather, get_exchange_rate])

 [Turn 1] get_weather, {'city': '서울'} -> 서울: 정보 없음
 [Turn 1] get_exchange_rate, {'from_currency': 'KRW', 'to_currency': 'JPY'} -> 1 KRW = 0.11 JPY


In [ ]:
def tool_loop_with_log(query, tools, max_turns = 5): # 만약 최대 턴 초과라면
  tool_map = {t.name:t for t in tools}
  llm_t = llm.bind_tools(tools)
  messages = [HumanMessage(content = query)]
  call_log = []
  for turn in range(max_turns):
    response = llm_t.invoke(messages)
    messages.append(response)

    if not response.tool_calls:
      return {"answer" : response.content, "log": call_log} # 툴을 호출 안했으니, 그냥 리턴

    for tc in response.tool_calls:
      tool_obj = tool_map[tc['name']]
      result = tool_obj.invoke(tc['args']) if tool_obj else "도구 없음."
      call_log.append({
          f" [Turn {turn+1}] {tc['name']}, {tc['args']} -> {result}"
      })

      messages.append(ToolMessage(content = str(result), tool_call_id = tc['id']))
      #print(f" [Turn {turn+1}] {tc['name']}, {tc['args']} -> {result}")

    return {"answer": "최대 턴 초과", "log" : call_log}

In [ ]:
ans = tool_loop_with_log("서울 날씨 알려주고, 원-엔 환율도 알려줘", [get_weather, get_exchange_rate])
ans

{'answer': '최대 턴 초과',
 'log': [{" [Turn 1] get_weather, {'city': '서울'} -> 서울: 정보 없음"},
  {" [Turn 1] get_exchange_rate, {'from_currency': 'KRW', 'to_currency': 'JPY'} -> 1 KRW = 0.11 JPY"}]}

In [ ]:
class WeatherReport(BaseModel):
  city: str = Field(description = 'city name')
  temperatrue : int = Field(description = 'temperature')
  condition: str = Field(description = 'weathre cond')
  recommendation : str = Field(description = 'activity recommend')

structured_llm = llm.with_structured_output(WeatherReport) # with_structured_output !!!!
report = structured_llm.invoke("서울의 봄 날씨를 보고서로 작성해줘")

report

WeatherReport(city='서울', temperatrue=17, condition='맑음', recommendation='야외에서 꽃구경하기')

In [1]:
llm_forced = llm.bind_tools([get_weather, calculate], tool_choice = 'get_weather')
resp = llm_forced.invoke("안녕하세요") # "안녕하세요" 라고, 날씨와 전혀 관련없는 문구를 줘도, Tool을 호출함
resp.tool_calls

NameError: name 'llm' is not defined

In [ ]:
structured_llm = llm.with_structured_output(WeatherReport)

In [ ]:
query = "안녕? 반가워"
messages = [HumanMessage(content = query)]

resp = llm_forced.invoke(messages)
messages.append(resp)

if resp.tool_calls:
  tc = resp.tool_calls[0]
  tool_result = get_weather.invoke(tc['args'])

  messages.append(ToolMessage(content = str(tool_result), tool_call_id = tc['id']))

final_report = structured_llm.invoke(messages)

In [ ]:
final_report # "query = "안녕? 반가워" 였음.. WeatherReport라는 포맷에 맞추기 위해 아무 값이나 넣은 것임..

WeatherReport(city='Seoul', temperatrue=0, condition='정보 없음', recommendation='야외 활동을 피하고 실내에서 안전하게 지내세요.')

In [ ]:
class ETFAnalysis(BaseModel):
  etf_name:str = Field(description = 'ETF 상품명')
  risk_level : str = Field(description = '위험 등급')
  expected_return: float = Field(description = "예상 연간 수익률")
  recommendation : str = Field(description="투자 추천 의견")

etf_llm = llm.with_structured_output(ETFAnalysis)
analysis = etf_llm.invoke("KODEX S&P500TR은 미국 S&P500 지수를 추종하는 ETF 입니다. 운용 보수는 0.05%, 최근 수익률은 25.3%입니다. 분석해주세요.")

In [ ]:
analysis

ETFAnalysis(etf_name='KODEX S&P500TR', risk_level='중간')